# A549 MVP — Direction A (structure-based)

**Claim:** given a few A549 hits, PhenoCompass returns the *truly related*
compounds (highest score), including **structurally independent** chemotypes.
Runs today with only the checkpoints (structure tower + U2OS MoA anchors); no
morphology input. The A549 image→DINO morphology test is Direction B
(`a549_mvp/ROADMAP_B_morphology.md`).

Two demos: **(1) cross-context MoA recovery** and **(2) customer-anchor Expand**.

> `MOCK=True` runs without checkpoints (synthetic, watermarked). Set `False` and
> fill in the paths for real results.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("a549_mvp"))
import numpy as np, pandas as pd
import mvp_core as core, moa_mapping as mm, plots
from IPython.display import Image

MOCK            = True
FINAL_MODEL_DIR = "./final_model"
A549_COMPOUNDS  = "./a549_compounds.csv"     # from prepare_a549_compounds.py
ANCHOR_MODALITY = "joint"
OUT_DIR         = "./a549_mvp_out"; os.makedirs(os.path.join(OUT_DIR,"figures"), exist_ok=True)
MOA = mm.ANCHOR_MOAS; MOA

## 1. A549 compounds (SMILES + MoA)

In [ ]:
from run_mvp import load_compounds, make_synthetic_compounds
cdf = make_synthetic_compounds() if (MOCK and not os.path.exists(A549_COMPOUNDS)) else load_compounds(A549_COMPOUNDS)
print(len(cdf), "compounds |", mm.summarize_mapping(cdf["moa_raw"]))
smiles = cdf["smiles"].tolist(); true_moa = list(cdf["moa_cluster"]); cdf.head()

## 2. Load model (real or mock)

In [ ]:
if MOCK:
    from mock_model import MockPhenoCompass
    pc = MockPhenoCompass(MOA); pc.prime_true_moa(smiles, true_moa)
else:
    from phenocompass import PhenoCompass
    pc = PhenoCompass(anchor_modality=ANCHOR_MODALITY, device="cpu", final_model_dir=FINAL_MODEL_DIR)

## 3. Demo 1 — cross-context MoA recovery
Score each A549 structure against the six U2OS MoA anchors. Strong **diagonal** =
own MoA scores highest.

In [ ]:
raw = pc.score_against_anchors(smiles, moa_list=MOA)
anchor_scores = core.strip_score_suffix(raw); anchor_scores.index = range(len(anchor_scores))
cross = core.evaluate_cross_line(anchor_scores, true_moa, MOA, top_k=1)
print(f"macro AUROC={cross.macro_auroc:.3f}  rank-1 acc={cross.accuracy:.3f}  enrichment={cross.topk_enrichment:.1f}x")
plots.plot_score_heatmap(cross.score_df, MOA, os.path.join(OUT_DIR,"figures","nb_heatmap"),
    "A549 structure -> MoA anchor score (diagonal = correct)", mock=MOCK)
Image(os.path.join(OUT_DIR,"figures","nb_heatmap.png"))

## 4. Demo 2 — customer-anchor Expand
A few A549 hits of a MoA -> rank all other A549 compounds. Same-MoA should rank top.

In [ ]:
lat = core.ensemble_average(pc.compute_struct_embeddings(smiles))
retr = core.evaluate_fewshot_retrieval(lat, true_moa, MOA, n_anchor=3, k=10)
print(f"macro retrieval AUROC={retr.macro_auroc:.3f}  precision@10={retr.mean_precision_at_k:.3f}")
retr.per_moa

## 5. Structural independence (novel chemotype)
For correctly-recovered compounds: anchor score vs. max Tanimoto to the anchors.
Top-left = high MoA score at low structural similarity = independent chemotype.
*(Needs RDKit + real SMILES; skipped in the synthetic mock.)*

In [ ]:
def anchor_smiles(moa):
    return [str(s) for s in next(iter(pc.anchor_embeddings_dict[moa].values())).index]
lab = cross.score_df[cross.score_df.true_moa.notna()]
idx = lab.index.to_numpy()
score_true = np.array([lab.loc[i, lab.loc[i,"true_moa"]] for i in idx])
correct = (lab.true_moa==lab.pred_moa).to_numpy()
mt = np.full(len(idx), np.nan)
for m in MOA:
    sel = lab.true_moa.to_numpy()==m
    if sel.sum(): mt[sel] = core.max_tanimoto_to_refs([smiles[idx[j]] for j in np.where(sel)[0]], anchor_smiles(m))
if np.isfinite(mt).any():
    plots.plot_score_vs_tanimoto(mt, score_true, correct, os.path.join(OUT_DIR,"figures","nb_indep"),
        "Independent chemotypes: high MoA score at low Tanimoto", mock=MOCK)
    from IPython.display import Image as I2; display(I2(os.path.join(OUT_DIR,"figures","nb_indep.png")))
else:
    print("Tanimoto unavailable (mock / no RDKit).")

## 6. Takeaway
- Cross-context MoA recovery + Expand work on the A549 compound set, from
  structure alone → the PhenoBridge search/Expand loop.
- The independence plot shows phenotype/MoA matches that are structurally novel.
- Re-run with `MOCK=False` for real numbers; Direction B adds the true A549
  morphology cross-line test.